# Território Vivo — segurança hídrica no Cantareira

Análise com observações reais oficiais SABESP. Este notebook explora a qualidade dos dados, executa aprendizado supervisionado com separação temporal purgada, compara baselines e publica limitações. A saída entregue registra o snapshot de 09/11/2026. Não há dados sintéticos de treinamento.

**Pergunta:** quanto será o volume útil do Cantareira sete dias depois da última observação, e qual erro foi observado em um teste futuro preservado? A resposta não autoriza restrições de água nem descreve potabilidade ou acesso por bairro.

In [1]:
from pathlib import Path
import sys, json
from datetime import date
module_dir = Path.cwd()
if not (module_dir / 'source.py').exists():
    module_dir = module_dir / 'modules' / 'water'
assert (module_dir / 'source.py').exists(), 'Abra a partir da raiz do projeto ou de modules/water.'
sys.path.insert(0, str(module_dir.resolve()))
from source import ROOT, load_rows
rows, quality = load_rows()
print(json.dumps({k: quality[k] for k in ['rows','firstObservedAt','lastObservedAt','missingDates','nullFields','normalizedSha256']}, ensure_ascii=False, indent=2))


{
  "rows": 3235,
  "firstObservedAt": "2018-01-01",
  "lastObservedAt": "2026-11-09",
  "missingDates": [],
  "nullFields": {
    "volumePct": 0,
    "rainMm": 0,
    "inflowM3s": 0,
    "withdrawalM3s": 0,
    "downstreamM3s": 0,
    "volumeHm3": 0
  },
  "normalizedSha256": "7503226feae75cf5b69c910b93483b441a43978528718ea8addfeadbd78af01c"
}


## Proveniência e limpeza

Fonte: [Portal dos Mananciais SABESP](https://mananciais.sabesp.com.br/), sistema 64. O manifesto contém URLs concretas, coleta UTC e SHA-256; respostas brutas não foram alteradas. Licença específica de reutilização não localizada: acesso público não equivale a licença aberta.

O pipeline rejeita datas futuras ao corte, NaN, duplicações, sistema diferente e divergência entre resumo atual/série. Não interpola lacunas e não troca ausências por zero. Campos: volume útil (%/hm³), chuva (mm), afluência/captação/descarga (m³/s). O modelo não confunde todos os reservatórios RMSP com o Cantareira.

In [2]:
import numpy as np
volume = np.asarray([row['volumePct'] for row in rows])
rain = np.asarray([row['rainMm'] for row in rows])
print('Volume: mínimo, quartis, máximo (%) =', np.round(np.quantile(volume, [0,.25,.5,.75,1]), 3).tolist())
print('Chuva média diária (mm) =', round(float(rain.mean()), 3))
print('Dias sem chuva =', int((rain == 0).sum()))
print('Correlação exploratória chuva/variação contemporânea =', round(float(np.corrcoef(rain[1:], np.diff(volume))[0,1]), 3))
print('Correlação é associação exploratória; não é causalidade nem evidência de previsão futura.')


Volume: mínimo, quartis, máximo (%) = [19.289, 38.278, 45.846, 57.709, 85.905]
Chuva média diária (mm) = 3.606
Dias sem chuva = 1591
Correlação exploratória chuva/variação contemporânea = 0.623
Correlação é associação exploratória; não é causalidade nem evidência de previsão futura.


## Construção do alvo e separação temporal

O modelo aprende ΔV = V(t+7) − V(t); a previsão final soma ΔV ao volume conhecido em t. Entradas usam somente t e passado: 14 dias de histórico, chuva e vazão observadas e sazonalidade anual. Ausência de chuva futura e operação física detalhada limita a projeção.

Partições cronológicas 60% treino, 15% validação, 10% calibração e 15% teste. Todas as datas do histórico e alvo de uma amostra devem caber na mesma partição; 21 origens são excluídas por fronteira. A validação escolhe parâmetros; calibração constrói o intervalo; teste mede desempenho uma única vez, sem escolher/refazer o modelo.

In [3]:
from modeling import samples, split_samples, FEATURES
parts = split_samples(rows, samples(rows))
print('Features:', ', '.join(FEATURES))
for name, items in parts.items():
    print(name, len(items), 'janelas;', items[0]['inputStart'], 'até', items[-1]['targetDate'])
for left, right in zip(list(parts.values()), list(parts.values())[1:]):
    assert max(s['targetDate'] for s in left) < min(s['inputStart'] for s in right)
print('Isolamento temporal confirmado: nenhum dia bruto compartilhado entre partições.')


Features: volume_t, volume_t_minus_1, volume_t_minus_7, volume_t_minus_14, change_1d, change_7d, change_14d, rain_7d_mm, rain_14d_mm, inflow_7d_m3s, withdrawal_7d_m3s, annual_sin, annual_cos
train 1920 janelas; 2018-01-01 até 2023-04-25
validation 464 janelas; 2023-04-26 até 2024-08-22
calibration 302 janelas; 2024-08-23 até 2025-07-11
test 465 janelas; 2025-07-12 até 2026-11-09
Isolamento temporal confirmado: nenhum dia bruto compartilhado entre partições.


## Treinamento real e seleção

São treinados quatro Ridge com padronização ajustada só no treino, e duas Random Forest de 180 árvores com semente 42. O menor MAE na validação seleciona o modelo. Persistência e deriva de sete dias usam os mesmos alvos de teste. Reexecutar a célula abaixo retreina com o snapshot atual do CSV e regenera os artefatos; não consulta a rede.

O modelo permanece congelado após o treino. Dados recentes de teste servem para medir o resultado, não para ajustar coeficientes. Os JSONs publicados indicam até quando houve treinamento.

In [4]:
from modeling import run
dashboard = run(as_of=date.fromisoformat(quality['requestedAsOf']))
evaluation = dashboard['evaluation']
print('Selecionado:', evaluation['selectedModel'])
print('Ranking na validação (MAE p.p.):')
for candidate in evaluation['candidates']:
    print(candidate['model'], candidate['mae'])
print('Teste:', json.dumps({k:evaluation[k] for k in ['test','baseline','drift','relativeMaeImprovementVsPersistence']}, ensure_ascii=False, indent=2))


Selecionado: ridge_alpha_10
Ranking na validação (MAE p.p.):
ridge_alpha_10 0.544939
ridge_alpha_1 0.548234
ridge_alpha_100 0.549769
ridge_alpha_0.1 0.550051
random_forest_depth_6_leaf_10 0.686315
random_forest_depth_12_leaf_5 0.70951
Teste: {
  "test": {
    "mae": 0.717844,
    "rmse": 1.175562,
    "bias": -0.285522,
    "n": 465,
    "unit": "p.p. de volume útil"
  },
  "baseline": {
    "mae": 1.23119,
    "rmse": 1.710886,
    "bias": -0.16978,
    "n": 465,
    "unit": "p.p. de volume útil"
  },
  "drift": {
    "mae": 0.895677,
    "rmse": 1.438106,
    "bias": -0.051016,
    "n": 465,
    "unit": "p.p. de volume útil"
  },
  "relativeMaeImprovementVsPersistence": 0.416951
}


## Incerteza avaliada

O raio do intervalo nominal de 90% vem de resíduos absolutos da calibração com origens a cada sete dias, usando quantil superior de amostra finita. Dados hidrológicos não são permutáveis: não se afirma garantia de cobertura futura. A cobertura efetiva de teste é publicada, mesmo quando inferior à nominal. Não se usa o teste para inflar o intervalo e depois alegar avaliação independente.

`confidence: null` nos alertas evita transformar cobertura histórica em probabilidade individual de acerto. Toda decisão de alto impacto exige revisão humana.

In [5]:
print(json.dumps(evaluation['interval'], ensure_ascii=False, indent=2))
print('Previsão do snapshot:', json.dumps(dashboard['forecast'], ensure_ascii=False, indent=2))
if evaluation['coverage'] < .9:
    print('ATENÇÃO: intervalo abaixo da cobertura nominal no teste; incerteza subestimada neste período.')
from evaluate import evaluate
print('Auditoria independente:', json.dumps(evaluate(), ensure_ascii=False, indent=2))


{
  "method": "Raio do quantil superior de resíduos absolutos de calibração, origens a cada 7 dias",
  "nominalCoverage": 0.9,
  "radiusPp": 1.319009,
  "calibrationSamples": 44,
  "testCoverage": 0.83871,
  "nonOverlappingTestCoverage": 0.835821,
  "nonOverlappingTestSamples": 67,
  "meanWidthPp": 2.638018,
  "guarantee": "Sem garantia de cobertura futura: série temporal não permutável; cobertura publicada é empírica."
}
Previsão do snapshot: {
  "targetDate": "2026-11-16",
  "valuePct": 55.17871,
  "lowerPct": 53.8597,
  "upperPct": 56.49772,
  "horizonDays": 7,
  "model": "ridge_alpha_10",
  "intervalLevel": 0.9,
  "requiresHumanReview": true,
  "trainedThrough": "2023-04-25",
  "calibratedThrough": "2025-07-11",
  "publishedAfterEvaluation": true,
  "usableUntil": "2026-11-11",
  "expired": false
}
ATENÇÃO: intervalo abaixo da cobertura nominal no teste; incerteza subestimada neste período.
Auditoria independente: {
  "test": {
    "mae": 0.7178444989247311,
    "rmse": 1.175561814

In [6]:
from analyze import analyze
exploration = analyze()
print(json.dumps(exploration['byYear'], ensure_ascii=False, indent=2))
print('Figuras SVG salvas em', ROOT / 'artifacts' / 'figures')


[
  {
    "year": 2018,
    "days": 365,
    "meanVolumePct": 43.282,
    "rainSumMm": 1232.25
  },
  {
    "year": 2019,
    "days": 365,
    "meanVolumePct": 49.246,
    "rainSumMm": 1358.3
  },
  {
    "year": 2020,
    "days": 366,
    "meanVolumePct": 49.462,
    "rainSumMm": 1182.2
  },
  {
    "year": 2021,
    "days": 365,
    "meanVolumePct": 40.053,
    "rainSumMm": 1087.05
  },
  {
    "year": 2022,
    "days": 365,
    "meanVolumePct": 37.385,
    "rainSumMm": 1402.5
  },
  {
    "year": 2023,
    "days": 365,
    "meanVolumePct": 73.831,
    "rainSumMm": 1526.615
  },
  {
    "year": 2024,
    "days": 366,
    "meanVolumePct": 63.107,
    "rainSumMm": 1349.4
  },
  {
    "year": 2025,
    "days": 365,
    "meanVolumePct": 42.592,
    "rainSumMm": 1140.5
  },
  {
    "year": 2026,
    "days": 313,
    "meanVolumePct": 37.631,
    "rainSumMm": 1387.2
  }
]
Figuras SVG salvas em C:\Users\digoc\OneDrive\Documentos\GS2\Principal\modules\water\artifacts\figures


## Visualização dos dados e dos erros

![Volume real histórico](artifacts/figures/volume_historico.svg)

![Previsão e alvo real no teste](artifacts/figures/avaliacao_temporal.svg)

![Comparação contra baselines](artifacts/figures/comparacao_baselines.svg)

## Interpretação e uso responsável

No snapshot entregue, Ridge alpha10 supera persistência e deriva pelo MAE do teste. Isso só vale para o período e recorte avaliados. Há viés negativo e intervalo de 90% com cobertura efetiva inferior a 90%, evidência de limite do modelo e mudança de regime. A escala agregada não permite inferir desigualdade territorial, potabilidade, esgoto ou situação domiciliar.

Volume abaixo de um limiar é referência diária, não enquadramento operacional mensal, definido por ANA/SP-Águas. Consultar [Resolução ANA/DAEE925/2017](https://www.gov.br/ana/pt-br/legislacao/resolucoes/resolucoes-regulatorias/2017/925) e comunicados atuais. A previsão do protótipo vence após dois dias sem novas observações; não autoriza captação, redução de pressão ou distribuição automática.

Melhorias futuras exigem dados adicionais e avaliação prospectiva: chuva prevista com horário de disponibilidade, demanda/operação, diagnóstico de deriva, calibração móvel predefinida e validação temporal em novos períodos. Não apresentar ajuste posterior ao teste como resultado independente.

Para reproduzir a coleta sem rede: `python update.py --as-of 2026-11-09 --offline`. Testes de integridade e vazamento: `python -m unittest discover -s modules/water -p test_water.py -v` a partir da raiz; comandos completos no README.